# Day 041 — Exercise 5: ask_df

**What you'll build:** `ask_df(df, question, model) -> str` — the full end-to-end pipeline: get schema, build prompt, call Ollama, extract code, exec, return result.

**Why it matters:** Five lines of composition wire together everything you built in Exercises 1-4. A user types a plain-English question, the LLM writes the pandas code, and you execute it and return the answer. That is 'Chat with your CSV' — no query language, no SQL, just English.

## Provided: All Pipeline Functions

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import ollama
import pandas as pd
import io


def get_df_schema(df) -> str:
    lines = [f"Shape: {df.shape[0]} rows x {df.shape[1]} columns"]
    lines.append("\nColumns and dtypes:")
    for col, dtype in df.dtypes.items():
        lines.append(f"  {col}: {dtype}")
    lines.append(f"\nSample (first 3 rows):\n{df.head(3).to_string(index=False)}")
    return "\n".join(lines)


def build_query_prompt(question: str, schema_str: str) -> str:
    return (
        "You are a Python data analyst. Write pandas code to answer the question.\n\n"
        "Requirements:\n"
        "- The DataFrame is already loaded as `df`. `pd` is also in scope.\n"
        "- Store the final answer in a variable named `result`.\n"
        "- Respond with ONLY a fenced Python code block, no explanation.\n\n"
        f"DataFrame schema:\n{schema_str}\n\n"
        f"Question: {question}"
    )


import re

def extract_code(response: str) -> str:
    fence = '`' * 3
    match = re.search(fence + r'python\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(fence + r'\s*(.*?)' + fence, response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()


import pandas as pd

def run_pandas_code(code: str, df) -> str:
    namespace = {'df': df, 'pd': pd}
    try:
        exec(code, namespace)
    except Exception as e:
        return f"Code execution error: {e}"
    result = namespace.get('result', 'No result variable found')
    return str(result)

In [ ]:
RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

## Your Implementation

In [ ]:
import re
import ollama
import pandas as pd

def ask_df(df, question: str, model: str = 'llama3.2') -> str:
    """
    Answer a plain-English question about a DataFrame using an LLM.

    Pipeline:
    1. schema  = get_df_schema(df)
    2. prompt  = build_query_prompt(question, schema)
    3. resp    = ollama.chat(model=model, messages=[{user: prompt}])
    4. code    = extract_code(resp['message']['content'])
    5. return run_pandas_code(code, df)

    Returns:
        str — the answer (result of the exec'd pandas code as a string)
    """
    # TODO: wire up the 5-step pipeline
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0
    _result = None

    # Check 1: function defined
    try:
        assert 'ask_df' in globals()
        passed += 1; print('\u2705 Check 1: ask_df is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string (Ollama call)
    try:
        _result = ask_df(SALES_DF, 'What is the total revenue?')
        assert isinstance(_result, str), \
            f'expected str, got {type(_result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: response is non-empty
    try:
        assert len(_result.strip()) > 0, 'response is empty'
        passed += 1; print(f'\u2705 Check 3: response is non-empty')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: total revenue answer contains 4105 (the correct sum)
    try:
        assert '4105' in _result, \
            f'total revenue should be 4105.0, got {repr(_result)}'
        passed += 1; print(f'\u2705 Check 4: correct revenue total in answer')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: also works for a different question
    try:
        _result2 = ask_df(SALES_DF,
                          'How many rows are in the dataset?')
        assert isinstance(_result2, str) and len(_result2.strip()) > 0
        passed += 1; print(f'\u2705 Check 5: works for different question ({repr(_result2[:30])})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import re
import ollama
import pandas as pd

def ask_df(df, question: str, model: str = 'llama3.2') -> str:
    schema  = get_df_schema(df)
    prompt  = build_query_prompt(question, schema)
    resp    = ollama.chat(model=model,
                          messages=[{"role": "user", "content": prompt}])
    code    = extract_code(resp["message"]["content"])
    return run_pandas_code(code, df)
```

</details>